# [실습1] Vector DB 캐싱

## 실습 목표
---
Vector DB 인덱싱 시간을 절약하기 위한 캐싱 기법을 사용해 봅니다.

## 실습 목차
---

1. **Vector DB 임베딩 캐싱:** Vector DB를 캐싱하고 저장 및 불러오는 기능을 구현합니다.

## 실습 개요
---
본격적으로 챗봇의 기능을 고도화 하기 전, 챗봇의 퀄리티를 높일 수 있는 다양한 방법을 학습합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [ ]:
import os
import time

from langchain_community.vectorstores import FAISS
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain.document_loaders import PyPDFLoader  

- Ollama를 통해 Mistral 7B 모델을 불러옵니다.

In [ ]:
!ollama pull mistral:7b

mistral:7b 모델을 사용하는 ChatOllama 객체를 생성합니다.

In [ ]:
llm = ChatOllama(model="mistral:7b")
route_llm = ChatOllama(model="mistral:7b", format="json")
embeddings = OllamaEmbeddings(model="mistral:7b")

data_dir = "data"

## 1. Vector DB 임베딩 캐싱

4장 실습3에서 저희는 시장 조사 문건을 불러와서 `OllamaEmbeddings`를 활용해 벡터로 변환하고, FAISS DB를 활용하여 저장했습니다.
- 출처: 한국소비자원의 2022년 키오스크(무인정보단말기) 이용 실태조사 보고서
  - https://www.kca.go.kr/smartconsumer/sub.do?menukey=7301&mode=view&no=1003409523&page=2&cate=00000057

In [ ]:
%%time

# 시장 조사 문건을 불러옵니다.
doc_path = os.path.join(data_dir, '키오스크(무인정보단말기) 이용실태 조사.pdf')
loader = PyPDFLoader(doc_path)
docs = loader.load()

vectorstore = FAISS.from_documents(
    docs,
    embedding=embeddings
)

db_retriever = vectorstore.as_retriever()

문서 하나를 불러오는데 약 2~3분 정도 소요되었습니다.<br>
만약 사용하고자 하는 문서가 매우 많다면 챗봇을 사용하려 할 때 마다 문서를 불러오면서 많은 시간이 낭비될 것입니다.

이를 방지하기 위해, 임베딩을 마친 Vector DB를 캐싱하는 방법과, 별도로 저장하는 방법을 학습해 봅시다.

### 1.1 임베딩 캐싱

VectorStore의 `save_local` 메서드를 활용해서 임베딩 완료된 DB를 별도의 파일로 추출할 수 있으며, `load_local` 메서드를 활용해서 다시 불러올 수 있습니다.

In [ ]:
vectorstore.save_local("./.cache/vectorstore/키오스크(무인정보단말기) 이용실태 조사")

In [ ]:
%%time

new_vectorstore = FAISS.load_local(
    "./.cache/vectorstore/키오스크(무인정보단말기) 이용실태 조사",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

db_retriever = new_vectorstore.as_retriever()

불러오는 시간이 크게 단축된 것을 확인할 수 있습니다.

`load_local` 메서드를 확인하면 `allow_dangerous_deserialization` 인자가 True로 설정되어 있습니다.

FAISS DB는 로컬 파일로 저장할 때 pickle을 사용합니다. pickle 라이브러리의 보안 취약성으로 인해, Product에는 임의의 사용자가 제공한 pkl 파일을 사용하지 않는 것을 강력히 권장합니다.<br> 즉, 개발자가 서버 단에 적용한 것이 확실한 파일만 불러오거나, pickle을 사용하지 않는 ChromaDB를 사용하는 등 보안 정책을 적용해야 합니다.

## 1.2 응답 캐싱

응답을 캐싱을 사용하여, 언어 모델의 응답을 저장하여 재사용할 수도 있습니다. 이를 통해 반복적인 질문에 대해 비용을 절약하고 응답 시간을 대폭 줄일 수 있습니다. 이렇게 하면 다음과 같은 이점이 있습니다:

- __비용 절약__: 동일한 질문에 대해 LLM을 반복 호출하지 않으므로 API 호출 비용을 절감할 수 있습니다.
- __빠른 응답__: 캐시에 저장된 결과를 즉시 반환할 수 있어, 응답 시간이 매우 빨라집니다.

In [ ]:
import time
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

set_llm_cache(InMemoryCache())

start_time = time.time()

response = llm.invoke("한국의 수도에 대해서 설명해줘.")

time_passed = time.time() - start_time

print(f"답변: {response.content}, 소요 시간: {round(time_passed, 2)} 초")

In [ ]:
start_time = time.time()

response = llm.invoke("한국의 수도에 대해서 설명해줘.")

time_passed = time.time() - start_time

print(f"답변: {response.content}, 소요 시간: {round(time_passed, 2)} 초")

LLM을 처음 호출할 때 대비 캐시를 사용해서 동일한 질문을 다시 물어보면 소요 시간이 크게 줄어드는 것을 확인할 수 있습니다.